# Inserts de Prueba

Importación de librerías

In [3]:
import random

from faker import Faker
from datetime import datetime, timedelta
from pathlib import Path

fake = Faker()

```sql
CREATE TABLE adverts (
    id INT UNSIGNED NOT NULL AUTO_INCREMENT,
    header VARCHAR(45) NOT NULL,
    description TEXT,
    created DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    published_from DATETIME NULL,
    published_to DATETIME NULL,
    visible BOOLEAN NOT NULL DEFAULT TRUE,
    section_id INT UNSIGNED NOT NULL,
    worker_id INT UNSIGNED NOT NULL
)
```

In [9]:
TOTAL = 1000

# Rango para published_from: 15 de marzo al 15 de diciembre de 2026
start_date = datetime(2026, 3, 15, 0, 0, 0)
end_date = datetime(2026, 12, 15, 23, 59, 59)
seconds_range = int((end_date - start_date).total_seconds())

text = "-- migrate:up\n\n"

for i in range(1, TOTAL + 1):

    # published_from aleatorio
    published_from = start_date + timedelta(
        seconds=random.randint(0, seconds_range)
    )

    # published_to entre 3 y 20 días después
    published_to = published_from + timedelta(
        days=random.randint(3, 20)
    )

    # created entre 0 y 30 días antes de published_from
    created = published_from - timedelta(
        days=random.randint(0, 30),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59),
    )

    # updated entre created y published_from
    diff_seconds = int((published_from - created).total_seconds())
    updated = created + timedelta(
        seconds=random.randint(0, diff_seconds)
    )

    # Texto aleatorio
    header = fake.sentence(nb_words=random.randint(3, 7)).replace("'", "''")
    description = fake.paragraph(nb_sentences=random.randint(3, 12)).replace("'", "''")

    visible = random.choice([0, 1])
    section_id = random.randint(1, 265)
    worker_id = random.randint(1, 30)

    text += f"""INSERT INTO adverts (id,header,description,created,updated,published_from,published_to,visible,section_id,worker_id) VALUES ({i},'{header}','{description}','{created:%Y-%m-%d %H:%M:%S}','{updated:%Y-%m-%d %H:%M:%S}','{published_from:%Y-%m-%d %H:%M:%S}','{published_to:%Y-%m-%d %H:%M:%S}',{visible},{section_id},{worker_id}
);\n"""

text += "-- migrate:down\n\nDELETE FROM adverts;\n"

# Crear carpeta tmp si no existe
output_dir = Path("..") / "tmp"
output_dir.mkdir(exist_ok=True)

# Nombre del archivo con timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = output_dir / f"inserts_adverts_{timestamp}.sql"

# Guardar archivo
with open(output_file, "w", encoding="utf-8") as archivo:
    archivo.write(text)

print(f"Archivo generado correctamente: {output_file}")

Archivo generado correctamente: ../tmp/inserts_adverts_20260717_200009.sql
